In [ ]:
import os
import certifi
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [6]:
from langchain.agents import create_react_agent, AgentExecutor

In [1]:
# Load the env variable

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

NameError: name 'certifi' is not defined

In [10]:
search_tool = TavilySearchResults(max_results=2)

In [ ]:
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city
    """

    url = (
        f"https://apiweatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"could not fetch weather data for {city}"
    
    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}C\n"
        f"Weather: {data['current']['weather_description'][0]}\n"
        f"Humidity : {data['current']['humidity']}%"
    )

In [14]:
result = search_tool.invoke("give me the latest news on ai")
result

[{'url': 'https://blog.google/innovation-and-ai/technology/ai/google-ai-updates-march-2026',
  'content': '## Bullet points\n\n "The latest AI news we announced in March" highlights Google\'s AI advancements across various products.\n Gemini got updates to better understand your context, turning devices into proactive, personalized helpers.\n Google Maps was upgraded with Gemini, offering conversational help and a redesigned navigation experience.\n New tools help you switch to Gemini, importing your chats and preferences from other AI apps.\n Google is expanding AI\'s role in healthcare with funding, partnerships, and Fitbit health tracking updates.\n\n## Basic explainer [...] Learn more:\n\nLearn more:\n\nLearn more:\n\nLearn more:\n\nLearn more:\n\nLearn more:\n\n# The latest AI news we announced in March 2026\n\nApr 01, 2026\n\nHere’s a recap of our biggest AI updates from March 2026, including an expansion of Search Live, more ways to access Personal Intelligence and new tools tha

In [16]:
# LLM

llm = ChatOpenAI(
    model = "gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [18]:
response = llm.invoke("what year is it?")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [19]:
# PROMPT

prompt = hub.pull("hwchase17/react")


c:\Users\thaku\anaconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [21]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [23]:
# TOOLS

tools = [search_tool]


In [24]:
# CREATE AGENT

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [25]:
# EXECUTER

agent_executer = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [26]:
# RUN

response = agent_executer.invoke({
    "input": (
        "Find the capital of INDIA"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [27]:
print(response["output"])

NameError: name 'response' is not defined